# Polymer + cell segmentation demo

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ugoreke/HbS-polymer-detection/blob/main/demo/demo.ipynb)

Loads `sample.jpg`, runs the frozen U-Net, segments cell instances via watershed, and renders
one figure with the raw image on top and the overlay (orange polymer tint + instance boundaries)
on the bottom.

The `sickling/` package is bundled with this notebook. The only external dependency is the
124 MB U-Net checkpoint, which is auto-downloaded from Google Drive on Colab (or pointed at a
local path otherwise).

In [ ]:
import os, sys, subprocess
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
MODEL_DRIVE_ID = '123OgOWBpMXkRRBDnfOsmkR1_-_MVgpF6'

if IN_COLAB:
    subprocess.run(['pip', '-q', 'install', 'gdown', 'scikit-image', 'h5py'], check=True)
    if not Path('sickling').exists():
        subprocess.run(['git', 'clone', '--depth', '1',
                        'https://github.com/ugoreke/HbS-polymer-detection.git', '/tmp/repo'], check=True)
        subprocess.run(['cp', '-r', '/tmp/repo/demo/sickling', '.'], check=True)
        subprocess.run(['cp', '/tmp/repo/demo/sample.jpg', '.'], check=True)
    MODEL_PATH = Path('unet_fold_2_best.pth')
    if not MODEL_PATH.exists():
        import gdown
        gdown.download(id=MODEL_DRIVE_ID, output=str(MODEL_PATH), quiet=False)
else:
    DEFAULT_LOCAL_MODEL = Path('/Users/utkugoreke/anaconda_projects/sickling/rbc-class/models/unet_fold_2_best.pth')
    MODEL_PATH = DEFAULT_LOCAL_MODEL if DEFAULT_LOCAL_MODEL.exists() else Path('unet_fold_2_best.pth')

sys.path.insert(0, str(Path('.').resolve()))
print(f'MODEL_PATH = {MODEL_PATH}')
assert MODEL_PATH.exists(), f'Missing model checkpoint at {MODEL_PATH}'

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy import ndimage as ndi

from sickling.config import load_config
from sickling.io.images import find_raw_image, load_raw_greyscale, normalize_image
from sickling.stage1_unet import load_unet, predict_label_map
from sickling.stage2_instances.watershed import mask_to_instances_with_reasons

SAMPLE_PATH = Path('sample.jpg')
POLYMER_RGB = np.array([255, 90, 0], dtype=np.uint8)
POLYMER_ALPHA = 0.5
BOUNDARY_RGB = np.array([90, 50, 255], dtype=np.uint8)
BOUNDARY_DILATE_ITERS = 2

In [ ]:
cfg = load_config()
model = load_unet(MODEL_PATH, n_classes=4)

In [ ]:
raw = load_raw_greyscale(SAMPLE_PATH)
raw_norm = normalize_image(raw)
label_map = predict_label_map(model, raw_norm, n_classes=4)
instance_image, stats, _, _ = mask_to_instances_with_reasons(
    label_map.astype(np.int64), cfg.instances, cfg.classes
)
print(f'cell instances kept: {stats.n_kept}/{stats.n_total}')

In [ ]:
# Greyscale -> RGB, alpha-blend orange where polymer, paint instance boundaries.
rgb = (np.stack([raw_norm] * 3, axis=-1) * 255).clip(0, 255).astype(np.uint8)

polymer_mask = (label_map == cfg.classes.polymer)
blend = (rgb[polymer_mask].astype(np.float32) * (1 - POLYMER_ALPHA)
         + POLYMER_RGB.astype(np.float32) * POLYMER_ALPHA)
rgb[polymer_mask] = blend.clip(0, 255).astype(np.uint8)

boundary_all = np.zeros(instance_image.shape, dtype=bool)
for iid in range(1, int(instance_image.max()) + 1):
    this = instance_image == iid
    if not this.any():
        continue
    dilated = ndi.binary_dilation(this, iterations=BOUNDARY_DILATE_ITERS)
    boundary_all |= (dilated & ~this)
rgb[boundary_all] = BOUNDARY_RGB

In [ ]:
h, w = raw_norm.shape
aspect = w / h
fig, axes = plt.subplots(2, 1, figsize=(8, 8 * 2 / aspect), sharex=True)
axes[0].imshow(raw_norm, cmap='gray', vmin=0, vmax=1)
axes[1].imshow(rgb)
for ax in axes:
    ax.set_xticks([])
    ax.set_yticks([])
fig.tight_layout()
plt.show()